# Remediation playbook — sl-demo (DHIS2 2.43.0.1)

**Replayable remediation playbook** — the executable companion to the written report.

### Launching this notebook
```bash
pip install jupyterlab httpx python-dotenv psycopg2-binary      # one-time
export DHIS2_BASE_URL="https://your-instance.org"                # NO /api suffix
export DHIS2_USER="admin"; export DHIS2_PASS="•••"               # or DHIS2_API_TOKEN
# SQL cells only: export PGHOST=… PGPORT=… PGDATABASE=… PGUSER=… PGPASSWORD=…
jupyter lab playbook.ipynb                                       # opens in your browser
```
(In a headless sandbox: add `--no-browser --ip 0.0.0.0 --port "$SANDBOX_HOST_PORT"` and open
`http://localhost:$SANDBOX_HOST_PORT` on the host. No Jupyter? `jupyter nbconvert --execute` runs it
headlessly — honours `DRY_RUN` — or copy the Setup cell + a fix cell into a plain `.py`.)

How to use:
1. Set the target via environment variables (`DHIS2_BASE_URL` + token or `DHIS2_USER`/`DHIS2_PASS`;
   for SQL cells also `PGHOST/PGPORT/PGDATABASE/PGUSER/PGPASSWORD`), then run the **Setup** cell.
2. With `DRY_RUN = True` (default), run cells to **preview** every action — nothing is written.
3. **Operator ack:** the Setup cell prints the target's system name. Verify it is the instance you mean
   to change (staging vs PROD!), copy that exact name into `CONFIRM_TARGET`, and re-run Setup. Non-DRY
   writes refuse to run until it matches — the drift guards catch the *wrong* database; this catches the
   right database at the wrong *time*.
4. Apply **one fix** (run its cell) or **everything** (Run All) by setting `DRY_RUN = False` and re-running.
5. Recommended flow: run on **staging** first, confirm the verify cells are clear, then repeat on **production**.

### SQL cells — two ways to run them
Most real deployments do **not** expose PostgreSQL to the machine running this notebook; usually only a
DBA on the server can run SQL. The Setup cell auto-detects which situation you are in (`SQL_MODE`):

- **`direct`** (PG env vars set — e.g. a local sandbox or a host with a DB route): SQL cells execute
  against the database directly, honouring `DRY_RUN`.
- **`server`** (no PG env vars): SQL cells do **not** attempt a connection. Each one prints its exact
  statement and its block reference in the accompanying **`sql-fixes-*.sql`** export — hand that block to
  the DBA (`psql -v ON_ERROR_STOP=1 -f`, or copy/paste one BEGIN..COMMIT block), confirm it ran, then
  continue with the next cell. Drive the ORDER from this notebook: it pauses at each SQL step exactly
  where the statement belongs in the fix sequence.

Either way, the verify cells afterwards prove the fix landed.

**Reading the output while a cell runs:** a line shows **⏳ … (waiting on API/DB)** the moment a write or
an integrity check starts, so a slow call never looks hung; it's replaced by **✅** (or **❌**) with the
HTTP status and **elapsed time** when the response returns. Integrity checks (`check(...)`) recompute
server-side and can take 30–120s — they print a **live "Ns elapsed" counter** while polling. A busy `[*]`
on the cell with no new ⏳ line just means the current call hasn't returned yet.

Every fix cell is **idempotent** (safe to re-run) and targets **UIDs** (portable across instances).
⚠️ Cells marked **SQL** require direct database access **and** the `ALL` authority on the target, run in
the order shown, and are followed by a cache-clear. They exist only where the DHIS2 API forbids the fix.


In [ ]:
# === Setup — run this first ===
# Target instance is taken from env vars, so this notebook is portable across staging/prod:
#   DHIS2_BASE_URL + (DHIS2_API_TOKEN | DHIS2_USER/DHIS2_PASS)
#   For SQL cells only: PGHOST PGPORT PGDATABASE PGUSER PGPASSWORD
import os, time, httpx

DRY_RUN = True   # <<< leave True to preview; set False to actually apply, then re-run the cells

CONFIRM_TARGET = ""   # <<< OPERATOR ACK: this cell prints the target's system name below — after
                      #     verifying it IS the instance you mean to change (staging vs PROD!), copy
                      #     that exact name here and re-run. Non-DRY writes refuse until it matches.
                      #     (The per-fix drift guards catch the WRONG database; this ack catches the
                      #     right database at the wrong TIME.)

BASE = os.environ["DHIS2_BASE_URL"].rstrip("/") + "/api"
_h = {"Accept": "application/json"}; _auth = None
if os.environ.get("DHIS2_API_TOKEN"): _h["Authorization"] = f"ApiToken {os.environ['DHIS2_API_TOKEN']}"
else: _auth = (os.environ.get("DHIS2_USER","admin"), os.environ.get("DHIS2_PASS","district"))
_client = httpx.Client(base_url=BASE, headers=_h, auth=_auth, timeout=300)

_info = {}
try: _info = _client.get("/system/info").json()
except Exception as _e: print("⚠️ could not read /system/info:", _e)
_SYSTEM_NAME = _info.get("systemName", "?")
try:
    _oucount = _client.get("/organisationUnits", params={"fields": "id", "pageSize": 1, "totalPages": "true"}).json()["pager"]["total"]
except Exception: _oucount = "?"
print(f"Target identity: systemName={_SYSTEM_NAME!r} | DHIS2 {_info.get('version','?')} | {BASE} | {_oucount} org units")
if CONFIRM_TARGET == _SYSTEM_NAME and _SYSTEM_NAME != "?":
    print("✅ operator ack: CONFIRM_TARGET matches — non-DRY writes are enabled.")
else:
    print(f"🔒 no operator ack yet — DRY_RUN preview works; before applying, set CONFIRM_TARGET = {_SYSTEM_NAME!r} and re-run this cell.")

def _ack():
    if DRY_RUN: return
    if CONFIRM_TARGET != _SYSTEM_NAME or _SYSTEM_NAME == "?":
        raise RuntimeError(f"OPERATOR ACK REQUIRED: verify this is the intended instance, then set "
                           f"CONFIRM_TARGET = {_SYSTEM_NAME!r} in the Setup cell and re-run it.")

# SQL execution mode — auto-detected, override by assigning SQL_MODE yourself and re-running this cell:
#   "direct": PG env vars present -> sql() runs statements against the DB (honours DRY_RUN)
#   "server": no DB route from here -> sql() prints the statement + its block in the sql-fixes-*.sql
#             export for a DBA to run on the server; this notebook then continues, assuming it was run.
SQL_MODE = "direct" if os.environ.get("PGHOST") else "server"
print("Target:", BASE, "| DRY_RUN =", DRY_RUN, "| SQL_MODE =", SQL_MODE,
      "" if SQL_MODE == "direct" else "(no PG env — SQL cells will defer to the sql-fixes-*.sql export)")

def _say(msg, end="\n"):
    # flush so the message appears IMMEDIATELY (Jupyter/terminal), not buffered until the cell ends.
    print(msg, end=end, flush=True)

def api(method, path, body=None, params=None):
    """HTTP to the target. GET always runs; writes honour DRY_RUN.
    Writes show a ⏳ 'waiting on API' line BEFORE the call (so a slow op never looks hung), then overwrite
    it with ✅/❌ + status + elapsed time when the response returns."""
    if method == "GET":
        return _client.get(path, params=params)
    if DRY_RUN:
        _say(f"  [DRY] {method} {path}" + (f"  body={body}" if body else "")); return None
    _ack()
    hdr = {"Content-Type": "application/json-patch+json"} if method == "PATCH" else None
    _say(f"  ⏳ {method} {path} … (waiting on API)", end="")
    t = time.time()
    r = _client.request(method, path, json=body, params=params, headers=hdr)
    mark = "✅" if r.status_code < 300 else "❌"
    _say(f"\r  {mark} {method} {path} → {r.status_code}  ({time.time()-t:.1f}s)" + " " * 12)
    return r

def cache_clear():
    return api("POST", "/maintenance", params={"cacheClear": "true"})

def check(name, timeout=120):
    """Fresh-run one data-integrity check on the target and return its issue count.
    The recompute is slow (often 30-120s), so this prints a LIVE elapsed counter while polling — it's
    obvious it's working, not hung."""
    before = (_client.get("/dataIntegrity/details", params={"checks": name}).json().get(name) or {}).get("finishedTime")
    _client.post("/dataIntegrity/details", params={"checks": name})
    t = time.time(); end = t + timeout
    _say(f"  ⏳ integrity check '{name}' recomputing (up to {timeout}s)…", end="")
    while time.time() < end:
        d = _client.get("/dataIntegrity/details", params={"checks": name}).json()
        if name in d and d[name].get("finishedTime") != before:
            n = len(d[name].get("issues", [])); _say(f"\r  \U0001f4ca check '{name}': {n} issue(s)  ({time.time()-t:.0f}s)" + " " * 20); return n
        _say(f"\r  ⏳ integrity check '{name}'… {int(time.time()-t)}s elapsed (polling)", end="")
        time.sleep(3)
    _say(f"\r  ⚠️ check '{name}': TIMED OUT after {timeout}s" + " " * 20); return None

# --- SQL helpers (only needed for guard-blocked fixes; require DB access + ALL on the target) ------
def _db():
    import psycopg2
    return psycopg2.connect(host=os.environ.get("PGHOST","localhost"), port=os.environ.get("PGPORT","5432"),
        dbname=os.environ.get("PGDATABASE","dhis2"), user=os.environ.get("PGUSER","dhis"),
        password=os.environ.get("PGPASSWORD",""))

def resolve_id(table, uid, idcol=None):
    """Map a stable UID to THIS instance's internal numeric id (portability — never hardcode ids).
    Requires a DB route — a code step using resolve_id() cannot run in SQL_MODE='server'; such fixes
    must either run where a route exists, or be re-expressed as dedicated sql steps with UID subqueries
    (which export to sql-fixes-*.sql)."""
    try:
        c = _db()
    except Exception as e:
        raise RuntimeError(f"resolve_id({table!r}, {uid!r}) needs a DB route and none is available ({e}). "
                           "Run this fix where PG env vars work, or use a sql step with UID subqueries.") from e
    cur = c.cursor()
    cur.execute(f"select {idcol or table+'id'} from {table} where uid=%s", (uid,))
    row = cur.fetchone(); c.close()
    return row[0] if row else None

def sql(stmt, args=None, note=""):
    """Run a write SQL on the target (honours DRY_RUN and SQL_MODE).
    SQL_MODE == "server": no DB route from this environment — print the statement and its block in the
    sql-fixes-*.sql export for a DBA to run on the server, then CONTINUE (the notebook assumes it was
    applied; the fix's verify cell will prove it either way). Always cache_clear() after metadata SQL."""
    if DRY_RUN:
        _say(f"  [DRY-SQL] {note}: {stmt}  args={args}"); return None
    if SQL_MODE == "server":
        _say("  🛢️  SQL (no DB route from here) — have a DBA run this on the server")
        _say("      (it is a ready-made BEGIN..COMMIT block in the accompanying sql-fixes-*.sql):")
        for line in stmt.strip().splitlines():
            _say("      " + line)
        if args:
            _say(f"      -- args: {args}  (statement in the .sql export has them inlined)")
        _say("      ▶ when it has been applied, continue with the next cell — the verify will confirm.")
        return None
    _ack()
    try:
        c = _db()
    except Exception as e:
        _say(f"  ❌ no DB connection ({e}) — set PG env vars for direct mode, or set SQL_MODE='server' "
             "and run the block from sql-fixes-*.sql on the server instead.")
        return None
    _say(f"  ⏳ SQL: {note} … (waiting on DB)", end="")
    t = time.time()
    cur = c.cursor(); cur.execute(stmt, args or ()); n = cur.rowcount; c.commit(); c.close()
    _say(f"\r  ✅ SQL: {note} → {n} row(s)  ({time.time()-t:.1f}s)" + " " * 12); return n

def new_uids(n=1):
    """Mint n valid DHIS2 UIDs from /api/system/id. Use ONCE while authoring to pre-generate a stable id
    for a NEW object, then HARD-CODE that id into the playbook so every run / every instance reuses it."""
    return _client.get("/system/id", params={"limit": n}).json()["codes"]

def ensure(path, uid, body):
    """Idempotent create-by-UID for a NEW object. If /<path>/<uid> exists -> skip; else create it with that
    EXPLICIT id. Pre-generating + hard-coding the id makes creation: re-run-safe (no duplicate), portable
    (same id on dedicated/staging/prod, so references resolve), and reversible (delete exactly that uid).
    `path` = plural endpoint, e.g. 'organisationUnitGroups'. Membership/derived fields go in `body`."""
    if _client.get(f"/{path}/{uid}").status_code == 200:
        _say(f"  ⏭️  exists, skip: /{path}/{uid}"); return uid
    if DRY_RUN:
        _say(f"  [DRY] create /{path}/{uid}  {body.get('name','')}"); return uid
    _ack()
    b = dict(body); b["id"] = uid
    _say(f"  ⏳ creating /{path}/{uid} … (waiting on API)", end="")
    t = time.time(); r = _client.post(f"/{path}", json=b)
    _say(f"\r  ✅ create /{path}/{uid} → {r.status_code}  ({time.time()-t:.1f}s)" + " " * 12); return uid


## [SEVERE] S1 — Merge duplicate category combo 'Morbidity Age' into 'Morbidity Cases' via the 2.43 merge endpoint, then dedupe the surviving combo's COCs  ·  _API_

- **Resolves check:** `categories_unique_category_combo`
- **Why:** Both combos use the same single category ('Morbidity Age'); one has no data elements. On 2.43 the categoryCombos/merge endpoint migrates dataset overrides, data values and COCs transactionally. Its own merge report warns duplicate COCs remain in the survivor — the second step clears those with categoryOptionCombos/merge.
- **Reversible:** Control snapshot; the merge migrates rather than deletes data.


In [ ]:
_skip = []
_n = check("categories_unique_category_combo")
if not _n: _skip.append("check categories_unique_category_combo already 0 (applied or never present)")
_r = api("GET", "/categoryCombos/pbvcDRasDav", params={"fields": "id"})
if _r is None or _r.status_code != 200: _skip.append("missing: /categoryCombos/pbvcDRasDav")
_r = api("GET", "/categoryCombos/t3aNCvHsoSn", params={"fields": "id"})
if _r is None or _r.status_code != 200: _skip.append("missing: /categoryCombos/t3aNCvHsoSn")
if _skip:
    _say("⛔ SKIPPED (drift guard): " + "; ".join(_skip))
    _say("   → the fix is already applied, or this instance differs from the one it was developed on. Review before forcing.")
else:
    api("POST", "/categoryCombos/merge", body={"sources": ["pbvcDRasDav"], "target": "t3aNCvHsoSn", "deleteSources": true, "dataMergeStrategy": "LAST_UPDATED"})  # 2.43+ only — on 2.42/2.41 this endpoint returns 405 and the fix needs the SQL merge recipe (see references/playbook.md §5/§6)
    # the combo merge leaves duplicate COCs in the survivor (per its own merge report) — dedupe them
    import collections
    cc = api("GET", "/categoryCombos/t3aNCvHsoSn", params={"fields": "categoryOptionCombos[id,name,categoryOptions[id]]"}).json()
    byopt = collections.defaultdict(list)
    for coc in cc.get("categoryOptionCombos", []):
        byopt[tuple(sorted(o["id"] for o in coc.get("categoryOptions", [])))].append(coc)
    for grp in (v for v in byopt.values() if len(v) > 1):
        target, sources = grp[0]["id"], [x["id"] for x in grp[1:]]
        _say(f"merging duplicate COCs {[x['name'] for x in grp[1:]]} -> {grp[0]['name']!r}")
        api("POST", "/categoryOptionCombos/merge", body={"sources": sources, "target": target, "deleteSources": True, "dataMergeStrategy": "LAST_UPDATED"})
    api("POST", "/maintenance", params={"cacheClear": "true", "categoryOptionComboUpdate": "true"})


### ✅ Verify — `categories_unique_category_combo` should now be 0

In [ ]:
n = check("categories_unique_category_combo")
assert n <= 0, f"REGRESSION: categories_unique_category_combo = {n} (> expected 0) — investigate"
print("OK" if n == 0 else f"{n} residual (expected 0, documented)")


### ✅ Verify — `category_option_combos_have_duplicates` should now be 0

In [ ]:
n = check("category_option_combos_have_duplicates")
assert n <= 0, f"REGRESSION: category_option_combos_have_duplicates = {n} (> expected 0) — investigate"
print("OK" if n == 0 else f"{n} residual (expected 0, documented)")


## [SEVERE] S2 — Fix the 'Births' combo's wrong category (Gender -> Location PHU/Community) — resolves 12 disjoint COCs, moves NO data  ·  _🛢️ SQL_

- **Resolves check:** `category_option_combos_disjoint`
- **Why:** The Births combo is defined as [Births attended by x Gender] but all 12 of its COCs actually use [Births attended by x Location PHU/Community] ('Trained TBA, In Community' etc.) — the combo references the wrong category while the data already sits under the correct COCs. No merge endpoint does a category swap and E1120 blocks the API edit while data exists, so this is the one legitimately-SQL structural fix even on 2.43.
- **Reversible:** Single-row UPDATE; swap back to the Gender category UID.
- **⚠️ Requires:** direct DB access + `ALL`. Run in order; a cache-clear follows.


In [ ]:
_skip = []
_n = check("category_option_combos_disjoint")
if not _n: _skip.append("check category_option_combos_disjoint already 0 (applied or never present)")
_r = api("GET", "/categoryCombos/m2jTvAj5kkm", params={"fields": "id"})
if _r is None or _r.status_code != 200: _skip.append("missing: /categoryCombos/m2jTvAj5kkm")
if not (any(c["id"] == "cX5k9anHEHd" for c in api("GET", "/categoryCombos/m2jTvAj5kkm", params={"fields": "categories[id]"}).json().get("categories", []))): _skip.append("custom precondition failed: the Births combo still (wrongly) references the Gender category — skip if alread")
if _skip:
    _say("⛔ SKIPPED (drift guard): " + "; ".join(_skip))
    _say("   → the fix is already applied, or this instance differs from the one it was developed on. Review before forcing.")
else:
    # NO DB ROUTE FROM HERE? This statement is block 'S2 (step 1/…)' in sql-fixes-sl-demo.sql — have a DBA run that block on the server, then continue below.
    sql("""DO $$ BEGIN
      IF NOT (EXISTS (SELECT 1 FROM categorycombos_categories cc JOIN categorycombo c ON c.categorycomboid=cc.categorycomboid JOIN category g ON g.categoryid=cc.categoryid WHERE c.uid='m2jTvAj5kkm' AND g.uid='cX5k9anHEHd')) THEN
        RAISE EXCEPTION 'GUARD FAILED for S2 (guard): database state differs from the pre-fix state this was built on';
      END IF;
    END $$;""", note="aborting guard — must pass before the step")
    sql("""UPDATE categorycombos_categories SET categoryid=(SELECT categoryid FROM category WHERE uid='x3uo8LqiTBk') WHERE categorycomboid=(SELECT categorycomboid FROM categorycombo WHERE uid='m2jTvAj5kkm') AND categoryid=(SELECT categoryid FROM category WHERE uid='cX5k9anHEHd')""", note="UID-based subqueries — never hardcode internal numeric ids")
    cache_clear()
    api("POST", "/maintenance", params={"cacheClear": "true", "categoryOptionComboUpdate": "true"})
cache_clear()


### ✅ Verify — `category_option_combos_disjoint` should now be 0

In [ ]:
n = check("category_option_combos_disjoint")
assert n <= 0, f"REGRESSION: category_option_combos_disjoint = {n} (> expected 0) — investigate"
print("OK" if n == 0 else f"{n} residual (expected 0, documented)")


## [SEVERE] S3 — Create an 'Other/unknown' group in each compulsory OU dimension (Facility Ownership, Facility Type, Location Rural/Urban); bucket unclassified OUs and move exclusive conflicts into it  ·  _API_

- **Resolves check:** `org_units_not_in_compulsory_group_sets`
- **Why:** The uncovered OUs are mostly admin-hierarchy levels with no facility classification; conflicted OUs carry contradictory memberships. An explicit Other/unknown bucket keeps each dimension complete and compulsory without fabricating classifications.
- **Reversible:** Delete the 3 groups by their fixed UIDs; original memberships are in the control snapshot.


In [ ]:
_skip = []
_n = check("org_units_not_in_compulsory_group_sets")
if not _n: _skip.append("check org_units_not_in_compulsory_group_sets already 0 (applied or never present)")
if not (len(api("GET", "/organisationUnitGroupSets", params={"filter": "name:eq:Facility Ownership", "fields": "id"}).json().get("organisationUnitGroupSets", [])) == 1): _skip.append("custom precondition failed: identity guard: the target has the dimension this fix was developed against")
if _skip:
    _say("⛔ SKIPPED (drift guard): " + "; ".join(_skip))
    _say("   → the fix is already applied, or this instance differs from the one it was developed on. Review before forcing.")
else:
    api("POST", "/metadata", body={"organisationUnitGroups": [{"id": "amN2FMJH1wj", "name": "Facility Ownership - Other/unknown", "shortName": "Ownership Other"}, {"id": "DVLpnxg8qmJ", "name": "Facility Type - Other/unknown", "shortName": "Facility Type Other"}, {"id": "Pn2mko7SfK1", "name": "Location Rural/Urban - Other/unknown", "shortName": "Location Other"}]}, params={"importStrategy": "CREATE_AND_UPDATE", "identifier": "UID", "atomicMode": "NONE"})  # fixed pre-generated UIDs: idempotent, portable, reversible; shells are also emitted to the import package
    GROUP_UID = {"Facility Ownership": "amN2FMJH1wj", "Facility Type": "DVLpnxg8qmJ", "Location Rural/Urban": "Pn2mko7SfK1"}
    import collections
    allou = {o["id"] for o in api("GET", "/organisationUnits", params={"fields": "id", "paging": "false"}).json()["organisationUnits"]}
    gss = api("GET", "/organisationUnitGroupSets", params={"fields": "id,name,compulsory,organisationUnitGroups[id]", "paging": "false"}).json()["organisationUnitGroupSets"]
    for gs in [g for g in gss if g.get("compulsory") and g["name"] in GROUP_UID]:
        ouid = GROUP_UID[gs["name"]]
        ou2grp = collections.defaultdict(list)
        for grp in gs["organisationUnitGroups"]:
            if grp["id"] == ouid: continue
            for o in api("GET", f"/organisationUnitGroups/{grp['id']}", params={"fields": "organisationUnits[id]"}).json().get("organisationUnits", []):
                ou2grp[o["id"]].append(grp["id"])
        uncovered = allou - set(ou2grp)
        conflicted = {o: g for o, g in ou2grp.items() if len(g) > 1}
        members = sorted(uncovered | set(conflicted))
        _say(f"{gs['name']}: uncovered={len(uncovered)} conflicted={len(conflicted)} -> Other gets {len(members)}")
        api("POST", "/metadata", body={"organisationUnitGroups": [{"id": ouid, "name": f"{gs['name']} - Other/unknown", "shortName": f"{gs['name'][:40]} Other"[:50], "organisationUnits": [{"id": m} for m in members]}]}, params={"importStrategy": "CREATE_AND_UPDATE", "identifier": "UID", "atomicMode": "NONE"})
        if ouid not in {g["id"] for g in gs["organisationUnitGroups"]}:
            api("POST", f"/organisationUnitGroupSets/{gs['id']}/organisationUnitGroups/{ouid}")
        for ou, grps in conflicted.items():
            for gid in grps:
                api("DELETE", f"/organisationUnitGroups/{gid}/organisationUnits/{ou}")


### ✅ Verify — `org_units_not_in_compulsory_group_sets` should now be 0

In [ ]:
n = check("org_units_not_in_compulsory_group_sets")
assert n <= 0, f"REGRESSION: org_units_not_in_compulsory_group_sets = {n} (> expected 0) — investigate"
print("OK" if n == 0 else f"{n} residual (expected 0, documented)")


## [WARNING] W1 — Repair validation rules wrapping a single operand in AVG() — strip the unsupported wrapper  ·  _API_

- **Resolves check:** `validation_rules_with_invalid_right_side_expression`
- **Why:** Validation-rule expressions do not support aggregation functions; AVG(x) over a single operand is a no-op wrapper, so stripping it is deterministic. Every candidate repair is validated server-side (GET /api/expressions/description) before writing; anything that does not validate is left and flagged.
- **Reversible:** Re-wrap in AVG() (original expressions in the control snapshot).


In [ ]:
_skip = []
_n = check("validation_rules_with_invalid_right_side_expression")
if not _n: _skip.append("check validation_rules_with_invalid_right_side_expression already 0 (applied or never present)")
if _skip:
    _say("⛔ SKIPPED (drift guard): " + "; ".join(_skip))
    _say("   → the fix is already applied, or this instance differs from the one it was developed on. Review before forcing.")
else:
    import re, time
    def valid(expr):
        r = api("GET", "/expressions/description", params={"expression": expr})
        try: return r.json().get("status") == "OK"
        except Exception: return False
    def strip_avg(e):
        e = (e or "").strip(); m = re.match(r"^AVG\((.*)\)$", e, re.S)
        return m.group(1).strip() if m else e
    d = api("GET", "/dataIntegrity/details", params={"checks": "validation_rules_with_invalid_right_side_expression"}).json().get("validation_rules_with_invalid_right_side_expression", {})
    for issue in d.get("issues", []):
        full = api("GET", f"/validationRules/{issue['id']}", params={"fields": ":owner"}).json()
        changed = False
        for side in ("leftSide", "rightSide"):
            e = full[side]["expression"]; ne = strip_avg(e)
            if ne != e and valid(ne):
                full[side]["expression"] = ne; changed = True
        if changed:
            api("PUT", f"/validationRules/{full['id']}", body=full)
            _say(f"repaired {full['name'][:60]!r}")
        else:
            _say(f"⏭️ no clean repair for {full['name'][:60]!r} — leave + flag")


### ✅ Verify — `validation_rules_with_invalid_right_side_expression` should now be 0

In [ ]:
n = check("validation_rules_with_invalid_right_side_expression")
assert n <= 0, f"REGRESSION: validation_rules_with_invalid_right_side_expression = {n} (> expected 0) — investigate"
print("OK" if n == 0 else f"{n} residual (expected 0, documented)")


## [SEVERE] U1 — Give users data-view access to their capture org units  ·  _API_

- **Resolves check:** `users_capture_ou_not_in_data_view_ou`
- **Why:** Users could enter data at org units they could not see in analytics; standard fix is data-view ⊇ capture.
- **Reversible:** Original dataViewOrganisationUnits are in the control snapshot.


In [ ]:
_skip = []
_n = check("users_capture_ou_not_in_data_view_ou")
if not _n: _skip.append("check users_capture_ou_not_in_data_view_ou already 0 (applied or never present)")
if _skip:
    _say("⛔ SKIPPED (drift guard): " + "; ".join(_skip))
    _say("   → the fix is already applied, or this instance differs from the one it was developed on. Review before forcing.")
else:
    us = api("GET", "/users", params={"fields": "id,username,organisationUnits[id],dataViewOrganisationUnits[id]", "paging": "false"}).json()["users"]
    for u in us:
        cap = {o["id"] for o in u.get("organisationUnits", [])}
        dv = {o["id"] for o in u.get("dataViewOrganisationUnits", [])}
        if cap and (cap - dv):
            api("PATCH", f"/users/{u['id']}", body=[{"op": "add", "path": "/dataViewOrganisationUnits", "value": [{"id": o} for o in sorted(cap | dv)]}])
            _say(f"  {u.get('username')}: data-view now covers capture")


### ✅ Verify — `users_capture_ou_not_in_data_view_ou` should now be 0

In [ ]:
n = check("users_capture_ou_not_in_data_view_ou")
assert n <= 0, f"REGRESSION: users_capture_ou_not_in_data_view_ou = {n} (> expected 0) — investigate"
print("OK" if n == 0 else f"{n} residual (expected 0, documented)")


## ⚠️ Flagged for owner — do NOT auto-fix
These need a decision from the metadata owner; they are deliberately not executable:

- **`data_elements_aggregate_with_different_period_types`** — 5 data elements live in datasets of different period types; which period is canonical is a reporting-design decision. *(owner: choose the canonical period type per data element)*
- **`category_option_group_sets_incomplete`** — 3 group sets flagged 'incomplete' — same-dimension banding overlap (options shared across overlapping categorizations), not fixable per-item without corrupting the dimension. *(owner: data-model review: stop sharing options across overlapping categorizations)*
- **`category_options_excess_groupset_membership`** — 'World Relief' donor option sits in BOTH the GIZ and USAID donor groups; neither is correct (it is a distinct donor). *(owner: assign the option to the right donor group (or its own))*
- **`data_elements_violating_exclusive_group_sets`** — 'Admission Date' is in two diagnosis groups; as a generic field it arguably belongs in neither. *(owner: decide the correct (single or no) diagnosis group)*
- **`tracked_entity_attributes_invalid_trigram_search_configuration`** — 56 TEAs searchable-but-unindexed; which to index is a per-needs performance decision. *(owner: curate the genuine search-key TEAs (names, IDs, phone) for indexing)*
- **`users_with_invalid_usernames`** — 1 invalid username; usernames are immutable via the API (E4056) and this is often a superuser. *(owner: confirm the replacement name; rename requires DB access)*